# EnergySense AI — Energy Consumption Analysis

**UCI Appliances Energy Prediction Dataset**

Candanedo, L. (2017). Appliances Energy Prediction. UCI Machine Learning Repository. DOI: 10.24432/C5VC8G

https://archive.ics.uci.edu/dataset/374/appliances%2Benergy%2Bprediction

---
This notebook walks through:
1. Dataset validation
2. Exploratory data analysis
3. Feature engineering
4. Model training and evaluation
5. Feature importance

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Dataset Validation

In [ ]:
from src.preprocessing import validate_dataset, load_and_clean

report = validate_dataset('../data/energydata_complete.csv')
for k, v in report.items():
    if k not in ('column_names', 'missing_per_column'):
        print(f'{k:30s}: {v}')

## 2. Load and Clean Dataset

In [ ]:
df = load_and_clean('../data/energydata_complete.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['Appliances'].plot(kind='hist', bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Appliance Consumption (Wh)')
axes[0].set_xlabel('Appliances (Wh)')

df.set_index('date')['Appliances'].resample('D').mean().plot(ax=axes[1], color='steelblue')
axes[1].set_title('Daily Average Appliance Consumption')
axes[1].set_ylabel('Wh (daily mean)')

plt.tight_layout()
plt.show()

print(df['Appliances'].describe())

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features, build_X_y, get_feature_columns

df_eng = engineer_features(df, include_lags=True)
print(f'Rows after lag warm-up: {len(df_eng):,}')
print(f'Total features: {len(get_feature_columns(df_eng))}')

X, y = build_X_y(df_eng)
print(f'X shape: {X.shape}, y shape: {y.shape}')

## 5. Hourly and Daily Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_eng.groupby('hour')['Appliances'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Average Consumption by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg Wh')

dow_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
df_eng.groupby('day_of_week')['Appliances'].mean().rename(index=dow_map).plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('Average Consumption by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Avg Wh')

plt.tight_layout()
plt.show()

## 6. Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

n = len(X)
split_idx = int(n * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

models = {
    'Linear Regression': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=30, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.02, max_depth=3, min_samples_leaf=30, subsample=0.8, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_test, pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2),
        'R2': round(r2_score(y_test, pred), 4),
    })

pd.DataFrame(results).sort_values('RMSE')

## 7. Actual vs Predicted

In [ ]:
best_model = models['Random Forest']
y_pred = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_show = 500
axes[0].plot(y_test.values[:n_show], label='Actual', alpha=0.7, color='steelblue')
axes[0].plot(y_pred[:n_show], label='Predicted', alpha=0.7, color='orange', linestyle='--')
axes[0].set_title(f'Actual vs Predicted (first {n_show} test obs)')
axes[0].set_ylabel('Appliances (Wh)')
axes[0].legend()

residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=60, color='purple', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=X.columns)
fi_top = fi.sort_values(ascending=False).head(20)

fi_top.sort_values().plot(kind='barh', color='steelblue', figsize=(10, 7))
plt.title('Top 20 Features by Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Note: Feature importance indicates predictive usefulness, not causality.')

---
## Summary

- **Dataset:** 19,735 observations at 10-minute intervals, Jan–May 2016. No missing values.
- **Target:** `Appliances` (Wh per 10-minute interval), range 10–1080 Wh.
- **Key finding:** Lag-1 (10 min past consumption) is the strongest predictor (importance ≈ 0.60).
- **Best model:** Random Forest — RMSE ≈ 57.7 Wh, R² ≈ 0.57 on chronological test set.
- **Evaluation:** Chronological split used to avoid temporal leakage.

---

**Citation:** Candanedo, L. (2017). Appliances Energy Prediction. UCI Machine Learning Repository. DOI: 10.24432/C5VC8G